In [ ]:
import pandas as pd
import json
import os
os.chdir("C:/Users/evere/blockchain-analytics")
os.getcwd()

ImportError: cannot import name 'load_metadata' from 'dashboard.services.transform' (C:\Users\evere\blockchain-analytics\dashboard\services\transform.py)

## Prices 

In [92]:
prices_raw = pd.read_json("data/prices.jsonl", lines=True)
prices_raw.head()

,timestamp,symbol,data
0,2025-12-23 21:19:04.047500,BNB,843.114787
1,2025-12-23 21:19:05.263572,LINK,12.404535
2,2025-12-23 21:19:06.571094,USDT,0.999400
3,2025-12-23 21:19:06.942851,USDC,0.999671
4,2025-12-23 21:19:08.520151,ETH,2963.975376


In [93]:
prices_raw["timestamp"] = pd.to_datetime(prices_raw["timestamp"])
prices_raw = prices_raw.rename(columns={"data": "price_USD"})
prices_raw = prices_raw.sort_values(by="timestamp")

prices_raw.head()



,timestamp,symbol,price_USD
0,2025-12-23 21:19:04.047500,BNB,843.114787
1,2025-12-23 21:19:05.263572,LINK,12.404535
2,2025-12-23 21:19:06.571094,USDT,0.999400
3,2025-12-23 21:19:06.942851,USDC,0.999671
4,2025-12-23 21:19:08.520151,ETH,2963.975376


In [121]:
from dashboard.services.transform import load_raw_prices, normalize_prices

raw = load_raw_prices()
prices = normalize_prices(raw)
prices.head()


,timestamp,symbol,price_USD
0,2025-12-23 21:19:04.047500,BNB,843.114787
1,2025-12-23 21:19:05.263572,LINK,12.404535
2,2025-12-23 21:19:06.571094,USDT,0.999400
3,2025-12-23 21:19:06.942851,USDC,0.999671
4,2025-12-23 21:19:08.520151,ETH,2963.975376


## Metadata

In [ ]:
metadata_df = pd.read_json("data/metadata.jsonl", lines=True)
metadata_df.head()

,timestamp,address,data
0,2025-12-27 17:18:45.678460,0xB8c77482e45F1F44dE1745F52C74426C631bDD52,"{'jsonrpc': '2.0', 'id': 1, 'result': {'decima..."
1,2025-12-27 17:18:46.393884,0x514910771AF9Ca656af840dff83E8264EcF986CA,"{'jsonrpc': '2.0', 'id': 1, 'result': {'decima..."
2,2025-12-27 17:18:47.213949,0xdAC17F958D2ee523a2206206994597C13D831ec7,"{'jsonrpc': '2.0', 'id': 1, 'result': {'decima..."
3,2025-12-27 17:18:48.032307,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,"{'jsonrpc': '2.0', 'id': 1, 'result': {'decima..."
4,2025-12-27 18:08:46.410077,0xB8c77482e45F1F44dE1745F52C74426C631bDD52,"{'jsonrpc': '2.0', 'id': 1, 'result': {'decima..."


In [96]:
metadata_df.iloc[0]["data"]

{'jsonrpc': '2.0',
 'id': 1,
 'result': {'decimals': 18,
  'logo': 'https://static.alchemyapi.io/images/assets/1839.png',
  'name': 'Binance Coin',
  'symbol': 'BNB'}}

In [97]:
decimal_list = metadata_df["data"].apply(lambda x: x["result"]["decimals"])
decimal_list.head()

0    18
1    18
2     6
3     6
4    18
Name: data, dtype: int64

In [98]:
token_symbol_list = metadata_df["data"].apply(lambda x: x["result"]["symbol"])
token_symbol_list.head()

0     BNB
1    LINK
2    USDT
3    USDC
4     BNB
Name: data, dtype: object

In [99]:
token_name_list = metadata_df["data"].apply(lambda x: x["result"]["name"])
token_name_list.head()

0    Binance Coin
1       Chainlink
2     Tether USDt
3            USDC
4    Binance Coin
Name: data, dtype: object

In [100]:
decmials = decimal_list.explode().reset_index(drop=True)
decmials = decmials.apply(pd.Series)
decmials["symbol"] = token_symbol_list
decmials["name"] = token_name_list

decmials.head()

,0,symbol,name
0,18,BNB,Binance Coin
1,18,LINK,Chainlink
2,6,USDT,Tether USDt
3,6,USDC,USDC
4,18,BNB,Binance Coin


In [101]:
decmials["address"] = metadata_df["address"]
decmials["decimals"] = decmials[0]
decmials[["address", "name", "symbol", "decimals"]].head()


,address,name,symbol,decimals
0,0xB8c77482e45F1F44dE1745F52C74426C631bDD52,Binance Coin,BNB,18
1,0x514910771AF9Ca656af840dff83E8264EcF986CA,Chainlink,LINK,18
2,0xdAC17F958D2ee523a2206206994597C13D831ec7,Tether USDt,USDT,6
3,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,USDC,USDC,6
4,0xB8c77482e45F1F44dE1745F52C74426C631bDD52,Binance Coin,BNB,18


## Balances

In [102]:
balances_raw = pd.read_json("data/balances.jsonl", lines=True)
balances_raw.head()

,timestamp,address,data
0,2025-12-23 21:19:03.023003,0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045,"{'jsonrpc': '2.0', 'id': 1, 'result': {'addres..."
1,2025-12-23 22:11:06.573313,0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045,"{'jsonrpc': '2.0', 'id': 1, 'result': {'addres..."
2,2025-12-25 19:47:34.918305,0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045,"{'jsonrpc': '2.0', 'id': 1, 'result': {'addres..."
3,2025-12-25 19:59:48.531582,0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045,"{'jsonrpc': '2.0', 'id': 1, 'result': {'addres..."
4,2025-12-27 00:51:39.885146,0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045,"{'jsonrpc': '2.0', 'id': 1, 'result': {'addres..."


In [103]:
balances_raw.iloc[0]["data"]

{'jsonrpc': '2.0',
 'id': 1,
 'result': {'address': '0xd8da6bf26964af9d7eed9e03e53415d37aa96045',
  'tokenBalances': [{'contractAddress': '0x514910771AF9Ca656af840dff83E8264EcF986CA',
    'tokenBalance': '0x00000000000000000000000000000000000000000000000018abf87e788f0800'},
   {'contractAddress': '0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48',
    'tokenBalance': '0x00000000000000000000000000000000000000000000000000000000f636d258'},
   {'contractAddress': '0xB8c77482e45F1F44dE1745F52C74426C631bDD52',
    'tokenBalance': '0x00000000000000000000000000000000000000000000000000470de4df820000'},
   {'contractAddress': '0xdAC17F958D2ee523a2206206994597C13D831ec7',
    'tokenBalance': '0x000000000000000000000000000000000000000000000000000000000ffc0d59'}]}}

In [104]:
token_lists = balances_raw["data"].apply(lambda x: x["result"]["tokenBalances"])
tokens = token_lists.explode().reset_index(drop=True)
tokens = tokens.apply(pd.Series)
tokens.head()

,contractAddress,tokenBalance
0,0x514910771AF9Ca656af840dff83E8264EcF986CA,0x00000000000000000000000000000000000000000000...
1,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,0x00000000000000000000000000000000000000000000...
2,0xB8c77482e45F1F44dE1745F52C74426C631bDD52,0x00000000000000000000000000000000000000000000...
3,0xdAC17F958D2ee523a2206206994597C13D831ec7,0x00000000000000000000000000000000000000000000...
4,0x514910771AF9Ca656af840dff83E8264EcF986CA,0x00000000000000000000000000000000000000000000...


In [105]:
def safe_hex_to_int(x):
    if isinstance(x, str) and x.startswith("0x"):
        return int(x, 16)
    return 0


In [106]:
tokens["tokenBalance"] = tokens["tokenBalance"].apply(safe_hex_to_int)
tokens.head()

,contractAddress,tokenBalance
0,0x514910771AF9Ca656af840dff83E8264EcF986CA,1777787700000000000
1,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,4130787928
2,0xB8c77482e45F1F44dE1745F52C74426C631bDD52,20000000000000000
3,0xdAC17F958D2ee523a2206206994597C13D831ec7,268176729
4,0x514910771AF9Ca656af840dff83E8264EcF986CA,1777787700000000000


In [107]:
tokens.tail()

,contractAddress,tokenBalance
23,0xdAC17F958D2ee523a2206206994597C13D831ec7,268176729
24,0x514910771AF9Ca656af840dff83E8264EcF986CA,1777787700000000000
25,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,4130787928
26,0xB8c77482e45F1F44dE1745F52C74426C631bDD52,20000000000000000
27,0xdAC17F958D2ee523a2206206994597C13D831ec7,268176729


In [108]:
tokens["timestamp"] = balances_raw["timestamp"].repeat(token_lists.apply(len)).values
tokens["wallet"] = balances_raw["address"].repeat(token_lists.apply(len)).values

tokens = tokens.rename(columns={
    "contractAddress": "contract_address",
    "tokenBalance": "raw_balance"
    })

tokens[["timestamp", "wallet", "contract_address", "raw_balance"]].head()

,timestamp,wallet,contract_address,raw_balance
0,2025-12-23 21:19:03.023003,0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045,0x514910771AF9Ca656af840dff83E8264EcF986CA,1777787700000000000
1,2025-12-23 21:19:03.023003,0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,4130787928
2,2025-12-23 21:19:03.023003,0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045,0xB8c77482e45F1F44dE1745F52C74426C631bDD52,20000000000000000
3,2025-12-23 21:19:03.023003,0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045,0xdAC17F958D2ee523a2206206994597C13D831ec7,268176729
4,2025-12-23 22:11:06.573313,0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045,0x514910771AF9Ca656af840dff83E8264EcF986CA,1777787700000000000


## Merge Metadata (Decimal) with Balance DF

In [109]:
def apply_token_decimals(tokens_df, decimals_df):
    merged_df = tokens_df.merge(
        decimals_df[["symbol", "name", "address", "decimals"]],
        left_on="contract_address",
        right_on="address",
        how="left"
    )

    merged_df["decimals"] = merged_df["decimals"].fillna(0).astype(int)
    merged_df["adjusted_balance"] = merged_df.apply(
        lambda row: row["raw_balance"] / (10 ** row["decimals"])
        if pd.notnull(row["decimals"]) else None,
        axis=1
    )
    
    return merged_df

In [110]:
merged_df = apply_token_decimals(tokens, decmials)
merged_df[["timestamp", "wallet", "contract_address", "symbol", "name", "raw_balance", "decimals", "adjusted_balance"]].head()

,timestamp,wallet,contract_address,symbol,name,raw_balance,decimals,adjusted_balance
0,2025-12-23 21:19:03.023003,0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045,0x514910771AF9Ca656af840dff83E8264EcF986CA,LINK,Chainlink,1777787700000000000,18,1.777788
1,2025-12-23 21:19:03.023003,0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045,0x514910771AF9Ca656af840dff83E8264EcF986CA,LINK,Chainlink,1777787700000000000,18,1.777788
2,2025-12-23 21:19:03.023003,0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,USDC,USDC,4130787928,6,4130.787928
3,2025-12-23 21:19:03.023003,0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,USDC,USDC,4130787928,6,4130.787928
4,2025-12-23 21:19:03.023003,0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045,0xB8c77482e45F1F44dE1745F52C74426C631bDD52,BNB,Binance Coin,20000000000000000,18,0.020000


### Test Transform